# Train/Test Split + Feature Building

This notebook:
1. Loads the raw Jigsaw dataset
2. Performs a **stratified** train/test split on `toxic` to control for class imbalance (~10% toxic)
3. Builds all dense features on the **train set only** (no leakage into test)
4. Saves the enriched train set to `01_data/01_processed/` for EDA

The test set is saved separately, untouched — features will be built on it later via `transform()` inside the sklearn Pipeline.

In [ ]:
import sys
import os

import pandas as pd
from sklearn.model_selection import train_test_split

# Make build_features importable from this notebook's location
sys.path.append(os.path.abspath("."))
from build_features import DenseFeatureTransformer

## 1. Load raw data

In [ ]:
DATA_PATH = "../../01_data/00_raw/jigsaw-dataset/train.csv/train.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows")
df.head(3)

In [ ]:
# Class balance check — toxic is the primary stratification column
toxic_rate = df["toxic"].mean()
print(f"Overall toxic rate: {toxic_rate:.1%}")
df["toxic"].value_counts()

## 2. Stratified train/test split

We stratify on `toxic` so both splits preserve the ~10% toxic rate.
80/20 split gives ~127K train rows and ~32K test rows.

In [ ]:
TARGET_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["toxic"],   # preserves toxic class ratio in both splits
)

# Reset indices so row numbers are clean
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"Train: {len(df_train):,} rows  |  toxic rate: {df_train['toxic'].mean():.1%}")
print(f"Test:  {len(df_test):,} rows   |  toxic rate: {df_test['toxic'].mean():.1%}")

## 3. Build dense features on train only

`DenseFeatureTransformer` is stateless (no vocabulary to learn), so `fit_transform` and `transform` are equivalent here. We still call `fit_transform` on train to stay consistent with the sklearn Pipeline convention — if we later swap this for a stateful transformer, the split is already correct.

In [ ]:
transformer = DenseFeatureTransformer()

print("Building features on train set...")
df_train_features = transformer.fit_transform(df_train)
print(f"Done. Shape: {df_train_features.shape}")

# Confirm all 32 feature columns were added
original_cols = set(df.columns)
new_cols = [c for c in df_train_features.columns if c not in original_cols]
print(f"\nNew feature columns ({len(new_cols)}):")
print(new_cols)

## 4. Quick EDA — feature distributions by toxicity label

In [ ]:
import matplotlib.pyplot as plt

# Mean feature values split by toxic / non-toxic
feature_means = (
    df_train_features
    .groupby("toxic")[new_cols]
    .mean()
    .T
    .rename(columns={0: "non_toxic", 1: "toxic"})
)

# Sort by the ratio of toxic/non-toxic mean (most discriminative first)
feature_means["ratio"] = feature_means["toxic"] / (feature_means["non_toxic"] + 1e-9)
feature_means = feature_means.sort_values("ratio", ascending=False)

feature_means[["non_toxic", "toxic"]].plot(
    kind="barh", figsize=(10, 12),
    title="Mean feature value: toxic vs non-toxic (train set)"
)
plt.xlabel("Mean value")
plt.tight_layout()
plt.show()

feature_means

## 5. Save enriched train set for teammate EDA

Saves to `01_data/01_processed/train_features.csv`.
Teammates can load this directly and skip the feature-building step.

In [ ]:
OUT_DIR   = "../../01_data/01_processed"
TRAIN_OUT = f"{OUT_DIR}/train_features.csv"
TEST_OUT  = f"{OUT_DIR}/test_raw.csv"

os.makedirs(OUT_DIR, exist_ok=True)

df_train_features.to_csv(TRAIN_OUT, index=False)
df_test.to_csv(TEST_OUT, index=False)

print(f"Saved train+features → {TRAIN_OUT}  ({len(df_train_features):,} rows, {df_train_features.shape[1]} cols)")
print(f"Saved raw test set   → {TEST_OUT}   ({len(df_test):,} rows)")